# CMSE 202 - Pneumonia Classification Using Traditional Machine Learning

## Pneumonia classifier base code

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.datasets import load_files
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report
from ipywidgets import interact

## Loading in data and organizing it

In [2]:
# Setup the directory paths
data_folder = os.path.join("..", "..", "data", "processed", "binary")
train_folder = os.path.join(data_folder, "train")
test_folder = os.path.join(data_folder, "test")

print(f"Train folder: {train_folder}")
print(f"Test folder: {test_folder}")

Train folder: ..\..\data\processed\binary\train
Test folder: ..\..\data\processed\binary\test


In [3]:
# Load in the images
train_dataset = load_files(train_folder, load_content=False)
test_dataset = load_files(test_folder, load_content=False)

X_train = [] 
X_test = []
y_train = train_dataset.target
y_test = test_dataset.target
categories = train_dataset.target_names

image_size = 128
grayscale = True

# Resize/grayscale the images and add them to respective arrays
for filename in train_dataset.filenames:
    try:
        img = img = Image.open(filename).convert("L" if grayscale else "RGB").resize((image_size, image_size))
        img_array = np.array(img).flatten()
        X_train.append(img_array)
    except Exception as e:
        print(f"Error loading image: {filename} - {e}")
        
for filename in test_dataset.filenames:
    try:
        img = img = Image.open(filename).convert("L" if grayscale else "RGB").resize((image_size, image_size))
        img_array = np.array(img).flatten()
        X_test.append(img_array)
    except Exception as e:
        print(f"Error loading image: {filename} - {e}")
        
X_train = np.array(X_train)
X_test = np.array(X_test)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (2000, 16384), Test shape: (500, 16384)


## Viewing data

In [4]:
# View the train dataset
def browse_images(images, labels, filenames, categories):
    n = len(images)
    def view_image(i):
        plt.imshow(images[i], cmap=plt.cm.gray, interpolation='nearest')
        plt.title(f'{categories[labels[i]]} ({filenames[i].split('\\')[-1]})')
        plt.axis('off')
        plt.show()
    interact(view_image, i=(0,n-1))
    
browse_images(X_train.reshape([X_train.shape[0], image_size, image_size]), y_train, train_dataset.filenames, categories)

interactive(children=(IntSlider(value=999, description='i', max=1999), Output()), _dom_classes=('widget-intera…

## Feature reduction

In [14]:
# PCA to preserve 95% variance in the data
variance = 0.95

pca = PCA(n_components=variance)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"PCA reduced features from {X_train.shape[1]} to {X_train_pca.shape[1]} capturing {variance*100}% variance")

PCA reduced features from 16384 to 329 capturing 95.0% variance


## Basic model

In [ ]:
# Basic model example
svm = SVC()
svm.fit(X_train_pca, y_train)
y_pred = svm.predict(X_test_pca)
print(classification_report(y_test, y_pred, target_names=categories))

              precision    recall  f1-score   support

      Normal       0.94      0.91      0.92       250
   Pneumonia       0.91      0.94      0.93       250

    accuracy                           0.93       500
   macro avg       0.93      0.93      0.93       500
weighted avg       0.93      0.93      0.93       500

